# JAX Communication-Bit Curriculum

This notebook trains communication-bit stages with the packaged JAX MAPPO runner and writes outputs under `runs/notebooks/communication_bits`.

Settings come from `experiments/communication_bits.json`, so the notebook stays a thin execution surface instead of carrying its own trainer or checkpoint conversion logic.


In [ ]:
from pathlib import Path
import os
import sys

# These must be set before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.65")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

{
    "project_root": PROJECT_ROOT,
    "jax_preallocate": os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"],
    "jax_memory_fraction": os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"],
}


In [ ]:
import sys

try:
    import jax  # noqa: F401
    import jax.numpy as jnp  # noqa: F401
    import tqdm  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "jax/notebook extras"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the JAX notebook extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[jax-cuda13,notebooks]"'
    ) from exc


In [ ]:
import jax
from tqdm.auto import tqdm

from ant_byte_env import MAX_WRITE_BITS
from ant_byte_env.experiments import config_args_to_argv, load_experiment_config
from ant_byte_env.rendering import render_checkpoint
from ant_byte_env.training.jax_mappo import main
from ant_byte_env.vault import create_vault_entry


## Curriculum Settings

Edit `experiments/communication_bits.json` for durable experiment changes. Override the derived values in this cell only for one-off notebook runs.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "communication_bits.json"
experiment = load_experiment_config(EXPERIMENT_CONFIG)
if experiment.backend != "jax":
    raise ValueError(f"Expected a JAX experiment config, got {experiment.backend!r}.")

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "communication_bits"
MEDIA_DIR = RUN_DIR / "media"
MEDIA_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_ARGS = dict(experiment.args)
BIT_STAGES = [int(bits) for bits in experiment.metadata.get("bit_stages", [TRAINING_ARGS.get("write_bits", 2)])]
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("global_update_cap", 2000))
NUM_ENVS = int(TRAINING_ARGS["num_envs"])
NUM_STEPS = int(TRAINING_ARGS["num_steps"])
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS
ROLLOUT_TILE_SIZE = 32

if any(bits <= 0 or bits > MAX_WRITE_BITS for bits in BIT_STAGES):
    raise ValueError(f"BIT_STAGES must contain integers from 1 to {MAX_WRITE_BITS}.")
if list(BIT_STAGES) != sorted(BIT_STAGES):
    raise ValueError("BIT_STAGES must be increasing.")

print(f"JAX device: {jax.devices()[0]}")
print(f"Experiment config: {EXPERIMENT_CONFIG}")
print(f"Communication bit stages: {BIT_STAGES}")


## Resolved Training Arguments

The shared config supplies the task, optimizer, and rollout settings. Each stage adds its own bit width, update budget, experiment name, and run directory.


In [ ]:
COMMON_ARGS = config_args_to_argv(
    {
        key: value
        for key, value in TRAINING_ARGS.items()
        if key not in {"exp_name", "write_bits", "total_timesteps", "save_model", "load_model", "run_dir"}
    }
)
COMMON_ARGS


## Train Bit Stages

Each stage trains through the packaged JAX runner and writes a current-format run directory. The default `GLOBAL_UPDATE_CAP` is intentionally set in the shared experiment config.


In [ ]:
stage_metrics = []
stage_checkpoint_paths = []

for target_bits in BIT_STAGES:
    stage_run_dir = RUN_DIR / f"{target_bits}_bits"
    checkpoint_path = stage_run_dir / "checkpoints" / "model.pkl"
    print(f"Training communication stage: {target_bits} writable bits")
    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{target_bits} bits",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_progress(update_index, total_updates, train_metrics):
        del total_updates
        update_iterator.update(1)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        stage_metrics.append(
            {
                "write_bits": target_bits,
                **train_metrics,
                "stage_update": update_index,
                "global_update_cap": GLOBAL_UPDATE_CAP,
                "checkpoint": str(checkpoint_path),
                "run_dir": str(stage_run_dir),
            }
        )

    train_args = [
        *COMMON_ARGS,
        "--exp-name", f"{experiment.args.get('exp_name', experiment.name)}_{target_bits}_bits",
        "--write-bits", str(target_bits),
        "--total-timesteps", str(UPDATE_TIMESTEPS * GLOBAL_UPDATE_CAP),
        "--run-dir", str(stage_run_dir),
    ]
    try:
        final_train_metrics = main(train_args, progress_callback=record_progress)
    finally:
        update_iterator.close()

    stage_checkpoint_paths.append(checkpoint_path)
    print(f"Saved {target_bits}-bit checkpoint to {checkpoint_path}")

FINAL_COMMUNICATION_CHECKPOINT = stage_checkpoint_paths[-1]
{
    "stage_checkpoint_paths": stage_checkpoint_paths,
    "final_checkpoint": FINAL_COMMUNICATION_CHECKPOINT,
    "final_train_metrics": final_train_metrics,
}


## Optional Render and Vault

Run this after training if you want rollout GIFs for the communication curriculum. It uses Pillow rather than ffmpeg so it does not fork a subprocess from the JAX kernel.

In [ ]:
def render_policy_rollout(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    rollout_path = MEDIA_DIR / f"{checkpoint_path.stem}_rollout.gif"
    return render_checkpoint(checkpoint_path, rollout_path, backend="jax")


In [ ]:
policy_checkpoint_paths = [
    RUN_DIR / f"{bits}_bits" / "checkpoints" / "model.pkl"
    for bits in BIT_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing communication stages before rendering:\n{missing}")

rollout_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering communication policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=RUN_DIR / "vault",
    title="JAX MAPPO communication-bit curriculum",
    description="Rollout GIFs for 15x15 JAX MAPPO policies trained with progressively larger writable communication alphabets.",
    assets=rollout_paths,
    metadata={
        "experiment_config": str(EXPERIMENT_CONFIG),
        "bit_stages": BIT_STAGES,
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "rollout_paths": [str(path) for path in rollout_paths],
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "rollout_paths": rollout_paths,
    "vault_entry_path": vault_entry_path,
}
